# Supply-chain pilot (TA feedback)
US companies must name, in their yearly 10-K filing, any customer that accounts for 10% or more of revenue.
This notebook downloads the 10-K of each of the 100 stocks in my universe for 2011, 2015, 2019 and 2023,
finds those "major customer" sentences, and checks whether the customer is also one of my 100 stocks.

**Run:** Runtime → Run all (no GPU needed). Takes about 15 to 25 minutes because the SEC allows only
10 downloads per second. Results go to `MyDrive/Connected-Markets/supply_chain/`. If Colab disconnects,
Run all again: finished companies are skipped.

In [ ]:
# ===== Supply-chain pilot: major-customer disclosures in 10-K filings (SEC EDGAR) =====
# For each of the 100 universe companies, download its 10-K for a few sample years, find sentences
# where it discloses a customer that accounts for a share of its revenue, and match that customer
# to the universe. Results are saved to Drive so a disconnect does not lose finished work.
import os, re, json, time, html as htmllib
import pandas as pd, requests

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/Connected-Markets'
except ImportError:
    ROOT = os.environ.get('CM_ROOT', './Connected-Markets')
OUT = f'{ROOT}/supply_chain'; os.makedirs(f'{OUT}/cache', exist_ok=True)
YEARS = [2011, 2015, 2019, 2023]                      # 10-K filing years sampled for the pilot
HEAD = {'User-Agent': 'Kabir Jain kabir.jain@mail.utoronto.ca', 'Accept-Encoding': 'gzip, deflate'}
OFFLINE = os.environ.get('CM_OFFLINE') == '1'          # only used to test this code without internet

def get(url):
    for attempt in range(4):
        try:
            r = requests.get(url, headers=HEAD, timeout=60)
            time.sleep(0.15)                               # SEC allows at most 10 requests per second
            if r.status_code == 200: return r
            if r.status_code in (403, 429): time.sleep(5 * (attempt + 1))
        except Exception:
            time.sleep(3 * (attempt + 1))
    return None

uni = pd.read_csv(f'{ROOT}/data/universe.csv')          # ticker, name, sector, sub_industry, CIK

# ---- names to look for in a disclosure sentence (current names plus older names and spellings) ----
EXTRA = {'WMT': ['wal-mart', 'walmart'], 'HPQ': ['hewlett-packard', 'hewlett packard', 'hp inc'],
         'GOOGL': ['google', 'alphabet'], 'ELV': ['anthem', 'wellpoint', 'elevance'],
         'RTX': ['raytheon', 'united technologies'], 'BNY': ['bank of new york', 'bny mellon'],
         'BKNG': ['priceline', 'booking holdings'], 'TGT': ['target corporation', 'target corp'],
         'DE': ['deere'], 'MMM': ['3m'], 'T': ['at&t'], 'GE': ['general electric'], 'F': ['ford motor'],
         'LOW': ["lowe's", 'lowes'], 'HD': ['home depot'], 'COST': ['costco'], 'BBY': ['best buy'],
         'CVS': ['cvs'], 'DIS': ['walt disney', 'disney'], 'UNH': ['unitedhealth'], 'FDX': ['fedex'],
         'PG': ['procter'], 'JNJ': ['johnson & johnson'], 'DD': ['dupont', 'du pont'],
         'CMCSA': ['comcast'], 'VZ': ['verizon'], 'IBM': ['ibm', 'international business machines'],
         'AMZN': ['amazon'], 'AAPL': ['apple'], 'MSFT': ['microsoft'], 'INTC': ['intel'], 'CSCO': ['cisco'],
         'BA': ['boeing'], 'LMT': ['lockheed'], 'CAT': ['caterpillar'], 'NKE': ['nike'],
         'XOM': ['exxon', 'exxonmobil'], 'SLB': ['schlumberger'], 'BMY': ['bristol-myers', 'bristol myers'],
         'ABT': ['abbott'], 'BAX': ['baxter'], 'PM': ['philip morris'], 'PNC': ['pnc'], 'AIG': ['aig'],
         'MA': ['mastercard'], 'KO': ['coca-cola'], 'MCD': ["mcdonald's"], 'NEM': ['newmont'],
         'SPG': ['simon property'], 'TMO': ['thermo fisher'], 'GS': ['goldman sachs'], 'MS': ['morgan stanley']}
SUFFIX = {'inc', 'inc.', 'corp', 'corp.', 'corporation', 'company', 'co', 'co.', 'plc', 'ltd', 'group',
          'holdings', 'incorporated', 'companies', '&', 'and', 'the', '(the)'}
NO_AUTO = {'TGT', 'F', 'GE', 'T', 'V', 'HPQ', 'BKNG'}       # names too ambiguous to search on their own
def aliases(row):
    words = str(row['name']).lower().replace(',', ' ').split()
    while words and words[0] in ('the',): words = words[1:]
    while words and words[-1] in SUFFIX: words = words[:-1]
    al = set(EXTRA.get(row['ticker'], []))
    name = ' '.join(words)
    if row['ticker'] not in NO_AUTO and len(name) >= 3: al.add(name)
    if row['ticker'] == 'V': al.update(['visa inc', 'visa u.s.a'])
    return sorted(al)
ALIAS = {r['ticker']: aliases(r) for _, r in uni.iterrows()}
PATTERNS = {t: re.compile(r'(?<![a-z])(' + '|'.join(re.escape(a) for a in al) + r')(?![a-z])')
            for t, al in ALIAS.items() if al}

# a "major customer" sentence: names a share of revenue or sales held by a customer
KEY = re.compile(r'(accounted for|represented|comprised|constituted)[^.]{0,120}?\d{1,2}(\.\d+)?\s?(%|percent)', re.I)
CTX = re.compile(r'revenue|net sales|sales|customer', re.I)

def text_of(doc):
    doc = re.sub(r'(?is)<(script|style).*?</\1>', ' ', doc)
    doc = re.sub(r'(?s)<[^>]+>', ' ', doc)
    return re.sub(r'\s+', ' ', htmllib.unescape(doc))

def disclosure_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+(?=[A-Z(])', text)
    return [s.strip() for s in sents if 40 < len(s) < 700 and KEY.search(s) and CTX.search(s)]

def ten_k_index(cik):
    path = f'{OUT}/cache/sub_{cik}.json'
    if os.path.exists(path): return json.load(open(path))
    r = get(f'https://data.sec.gov/submissions/CIK{int(cik):010d}.json')
    if r is None: return []
    js = r.json(); pages = [js['filings']['recent']]
    for f in js['filings'].get('files', []):
        rr = get(f"https://data.sec.gov/submissions/{f['name']}")
        if rr is not None: pages.append(rr.json())
    rows = []
    for p in pages:
        for form, date, acc, doc in zip(p['form'], p['filingDate'], p['accessionNumber'], p['primaryDocument']):
            if form in ('10-K', '10-K405'): rows.append((date, acc, doc))
    json.dump(rows, open(path, 'w'))
    return rows

def fetch_sentences(cik, year, rows):
    """Disclosure sentences from the first 10-K filed in `year` (None if there is no such 10-K)."""
    hits = sorted(r for r in rows if r[0].startswith(str(year)))
    if not hits: return None, None
    date, acc, doc = hits[0]
    path = f'{OUT}/cache/{cik}_{year}.json'
    if os.path.exists(path): return json.load(open(path)), date
    r = get(f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-', '')}/{doc}")
    if r is None: return None, date
    sents = disclosure_sentences(text_of(r.text))
    json.dump(sents, open(path, 'w'))
    return sents, date

records, status = [], []
if not OFFLINE:
    for i, row in uni.iterrows():
        rows = ten_k_index(row['CIK'])
        for Y in YEARS:
            sents, date = fetch_sentences(row['CIK'], Y, rows)
            status.append({'ticker': row['ticker'], 'year': Y, 'found_10k': sents is not None})
            if sents is None: continue
            for s in sents:
                matched = [t for t, p in PATTERNS.items() if t != row['ticker'] and p.search(s.lower())]
                records.append({'supplier': row['ticker'], 'year': Y, 'filed': date,
                                'customers_in_universe': ';'.join(matched), 'sentence': s[:600]})
        print(f'{i+1}/{len(uni)} {row["ticker"]:6s} done, {len(records)} disclosure sentences so far', end='\r')

    rec = pd.DataFrame(records, columns=['supplier', 'year', 'filed', 'customers_in_universe', 'sentence'])
    st = pd.DataFrame(status)
    rec.to_csv(f'{OUT}/disclosures.csv', index=False)
    edges = rec[rec.customers_in_universe != ''].copy()
    edges = edges.assign(customer=edges.customers_in_universe.str.split(';')).explode('customer')
    edges[['supplier', 'customer', 'year', 'filed', 'sentence']].to_csv(f'{OUT}/candidate_edges.csv', index=False)
    summary = {Y: {'10-Ks found': int(st[st.year == Y].found_10k.sum()),
                   'companies with a major-customer sentence': int(rec[rec.year == Y].supplier.nunique()),
                   'candidate in-universe edges': int(edges[edges.year == Y][['supplier', 'customer']].drop_duplicates().shape[0])}
               for Y in YEARS}
    json.dump(summary, open(f'{OUT}/summary.json', 'w'), indent=1)
    print('\n', pd.DataFrame(summary).T)
    print('\nCandidate in-universe edges (supplier -> customer):')
    print(edges[['year', 'supplier', 'customer']].drop_duplicates().to_string(index=False))
